In [249]:
import pandas as pd
import requests
import io

headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko)'}

# --- 1. GET PEOPLE DATA ---
url_people = 'https://en.wikipedia.org/wiki/Forbes_list_of_the_World%27s_Most_Powerful_People'
res_people = requests.get(url_people, headers=headers)
df_p = pd.read_html(io.StringIO(res_people.text))[1]

# Clean and format
people_list = df_p['Name'].str.replace(r'\[.*\]', '', regex=True).str.strip()
people_final = pd.DataFrame({'keyword': people_list, 'category': 'person'})

country_df = df_p[['Nationality']].rename(columns={'Nationality': 'keyword'})
country_df['category'] = 'geography'

# --- 2. GET COMPANY DATA ---
url_comp = 'https://en.wikipedia.org/wiki/List_of_largest_companies_by_revenue'
res_comp = requests.get(url_comp, headers=headers)
# Usually tables[0] is the main list, but we ensure we grab the 'Name' column
df_c = pd.read_html(io.StringIO(res_comp.text))[0]
df_c.columns = df_c.columns.get_level_values(-1) # Flatten MultiIndex

company_list = df_c['Name'].str.replace(r'\[.*\]', '', regex=True).str.strip()
company_final = pd.DataFrame({'keyword': company_list, 'category': 'company'})

# --- 3. GET STATE/GEOGRAPHY DATA ---
url_state = 'https://en.wikipedia.org/wiki/List_of_states_and_territories_of_the_United_States'
res_state = requests.get(url_state, headers=headers)
df_s = pd.read_html(io.StringIO(res_state.text))[1]
df_s.columns = df_s.columns.get_level_values(-1) # Flatten

df_states_raw = (
    df_s[['Flag, name and postal abbreviation[8]', 'Flag, name and postal abbreviation[8].1']]
    .melt(value_name='keyword')
    [['keyword']]
)

df_capitals = pd.DataFrame({
    'keyword': df_s['Capital'],
    'category': 'geography'
})

df_cities = pd.DataFrame({
    'keyword': df_s['Largest[12]'],
    'category': 'geography'
})


states_final = (
    pd.concat([df_states_raw.assign(category='geography'), df_capitals, df_cities], ignore_index=True)
    .replace(r'\[.*?\]', '', regex=True)
    .replace(r'[^\x00-\x7F]+', '', regex=True)
    .apply(lambda x: x.str.strip() if x.name == 'keyword' else x)
    .replace('', pd.NA)
    .dropna(subset=['keyword'])
    .drop_duplicates(subset=['keyword'])
    .reset_index(drop=True)
)

# --- 4. GET CRYPTO DATA ---
url_crypto = 'https://en.wikipedia.org/wiki/List_of_cryptocurrencies'
response = requests.get(url_crypto, headers=headers)

tables = pd.read_html(io.StringIO(response.text))
df_crypto = tables[0]
df_crypto.columns = df_crypto.columns.get_level_values(-1)


df_crypto_clean = (
    df_crypto[['Currency', 'Symbol']]
    .replace(r'\[.*?\]', '', regex=True) # Removes all [notes] and [123]
    .assign(Symbol=lambda x: x['Symbol'].str.split(r'[,/]\s*')) # Splits on comma or slash
    .explode('Symbol')
    .apply(lambda x: x.str.strip()) # Strips whitespace from all columns at once
)

df_crypto_final = (
    pd.melt(df_crypto_clean, value_name='keyword', var_name='source_col')
    [['keyword']]
    .drop_duplicates()
    .reset_index(drop=True)
)

df_crypto_final['category'] = 'crypto'


# --- 5. Months ---
months_dict = {
    "January": "month",
    "February": "month",
    "March": "month",
    "April": "month",
    "May": "month",
    "June": "month",
    "July": "month",
    "August": "month",
    "September": "month",
    "October": "month",
    "November": "month",
    "December": "month"
}

months_df = pd.DataFrame(list(months_dict.items()), columns=['keyword', 'category'])

# --- 6. Political ---
url_political = 'https://en.wikipedia.org/wiki/Political_parties_in_the_United_States'
response = requests.get(url_political, headers=headers)

tables = pd.read_html(io.StringIO(response.text))
df_pol = tables[3].T
df_pol = df_pol.reset_index()[['level_0']].rename(columns={'level_0': 'keyword'})
df_pol['keyword'] = df_pol['keyword'].str.replace(' Party', '')
df_pol['category'] = 'political'

political_terms = {
    "Senate": "political",
    "House of Representatives": "political",
    "Congress": "political",
    "President": "political",
    "Presidential": "political",
    "Supreme Court": "political",
    "Legislature": "political",
    "Judiciary": "political",
    "Executive": "political",
    "Independent": "political"
}
terms_df = pd.DataFrame(list(political_terms.items()), columns=['keyword', 'category'])

df_pol = pd.concat([terms_df], ignore_index=True)

# --- 6. Econ ---
econ_dict = {
    "Fed": "econ",
    "Federal Reserve": "econ",
    "GDP": "econ",
    "CPI": "econ",
    "Inflation": "econ",
    "Interest Rate": "econ",
    "Unemployment": "econ",
    "Recession": "econ",
    "Bull Market": "econ",
    "Bear Market": "econ",
    "Fiscal Policy": "econ",
    "Monetary Policy": "econ",
    "S&P 500": "econ",
    "Nasdaq": "econ",
    "Dow Jones": "econ",
    "Yield Curve": "econ",
    "Quantitative Easing": "econ",
}

econ_df = pd.DataFrame(list(econ_dict.items()), columns=['keyword', 'category'])


# --- 7. leaders and countries ---
leaders_url = 'https://en.wikipedia.org/wiki/List_of_current_heads_of_state_and_government'
response = requests.get(leaders_url, headers=headers)

tables = pd.read_html(io.StringIO(response.text))
df_raw = tables[1]

df_country = (
    df_raw[['State']]
    .rename(columns={'State': 'keyword'})
    .assign(category='geography')
    .replace(r'\[.*\]', '', regex=True)
    .apply(lambda x: x.str.strip())
    .drop_duplicates()
)

def extract_name(series):
    parts = series.str.split(r'[–-]', n=1, expand=True)
    return parts[1].fillna(parts[0]).str.strip()

df_leaders = (
    df_raw[['Head of state', 'Head of government']]
    .replace(r'.*?designate\s*[–-]\s*', '', regex=True)
    .apply(extract_name) # Your existing helper for splitting on dashes
    .replace(r'\[.*?\]', '', regex=True)
    .replace(r'\s+[IVXLCDM]+\b$', '', regex=True)
    .melt(value_name='keyword')
    [['keyword']] 
    .assign(category='person')
    .apply(lambda x: x.str.strip())
    .replace('', pd.NA)
    .dropna()
    .drop_duplicates()
)

# --- 8. Sports and Athletes ---
athletes_url = 'https://en.wikipedia.org/wiki/List_of_sports_figures_considered_the_greatest'
response = requests.get(athletes_url, headers=headers)

all_tables = pd.read_html(io.StringIO(response.text))

sport_dfs = []
for df in all_tables:
    df = df.rename(columns={'Athlete': 'player', 'Player': 'player', 'Name': 'player', 'Sport': 'sport'})
    if 'sport' in df.columns and 'player' in df.columns:
        sport_dfs.append(df[['sport', 'player']].copy())

df_raw_master = pd.concat(sport_dfs, ignore_index=True)

df_raw_master['player'] = (
    df_raw_master['player']
    .str.replace(r'\[.*?\]', '', regex=True) # Remove [xii]
    .str.split(r'[–-]', n=1).str[-1]        # Name to right of dash
    .str.replace(r'\s+[IVXLCDM]+\b$', '', regex=True, flags=re.IGNORECASE) # Frederik X
    .str.strip()
)

df_raw_master['sport'] = (
    df_raw_master['sport']
    .str.replace(r'\[.*?\]', '', regex=True) # Remove [12]
    .str.replace(r'\(.*?\)', '', regex=True) # Remove (general)
    .str.strip()
)

# --- SPORTS ---
junk_values = ['All sports', 'General', 'Miscellaneous', 'sports']
sport_df = (
    df_raw_master[['sport']]
    .rename(columns={'sport': 'keyword'})
    .loc[lambda df: ~df['keyword'].isin(junk_values)]
    .drop_duplicates()
    .assign(category='sport')
    .dropna()
)

# --- PLAYERS ---
player_df = (
    df_raw_master[['player']]
    .rename(columns={'player': 'keyword'})
    .drop_duplicates()
    .assign(category='person')
    .dropna()
)

final_sports_batch = pd.concat([sport_df, player_df], ignore_index=True)


# --- COMBINE EVERYTHING INTO ONE MASTER DF ---
master_df = pd.concat([people_final, company_final, 
                       states_final, country_df,

                       df_crypto_final, months_df,
                       df_pol, econ_df,
                       df_leaders, df_country,
                       final_sports_batch
                       
                       ], ignore_index=True)

master_df = master_df.dropna().drop_duplicates(subset=['keyword'])
master_df['keyword'] = master_df['keyword'].str.lower()

print(f"Total keywords recorded: {len(master_df)}")
master_df.sample(10)

Total keywords recorded: 1181


,keyword,category
75,walmart,company
914,guinea,geography
430,xpm,crypto
919,hungary,geography
160,oklahoma,geography
255,santa fe,geography
907,"gambia, the",geography
26,ma huateng,person
1249,brian mckeever,person
488,gdp,econ


In [250]:
contain_check = master_df[master_df['keyword'].str.contains('designate', case=False, na=False)]

print(f"Found {len(contain_check)} matches for Maduro:")
print(contain_check)

Found 0 matches for Maduro:
Empty DataFrame
Columns: [keyword, category]
Index: []


need more 
- top athletes

In [251]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
import re

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
# Adding your custom Wikipedia "junk" words
stop_words.update(['note', 'ref', 'total', 'list', 'world', 'people', 'of'])

# This splits the phrase, removes filler words, and puts it back together
def clean_phrase(text):
    if pd.isna(text): 
        return text
    # Split by any whitespace or punctuation if needed
    words = str(text).split()
    # Keep only words NOT in our stop_words set
    clean_words = [w for w in words if w.lower() not in stop_words]
    return " ".join(clean_words)

master_df_clean = master_df.copy()
master_df_clean['keyword'] = master_df_clean['keyword'].apply(clean_phrase)


master_df_clean = master_df_clean[master_df_clean['keyword'].str.strip() != ""]
master_df_clean = master_df_clean[master_df_clean['keyword'].str.len() > 1]
master_df_clean['keyword'] = master_df_clean['keyword'].str.replace(r'\s+', ' ', regex=True).str.strip()


master_df_clean.to_csv('data/keywords.csv', index=False)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/blakeuribe/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
